<a href="https://www.kaggle.com/code/avikdas567/image2biomass-prediction?scriptVersionId=291655991" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/csiro-biomass/sample_submission.csv
/kaggle/input/csiro-biomass/train.csv
/kaggle/input/csiro-biomass/test.csv
/kaggle/input/csiro-biomass/test/ID1001187975.jpg
/kaggle/input/csiro-biomass/train/ID2099464826.jpg
/kaggle/input/csiro-biomass/train/ID2037861084.jpg
/kaggle/input/csiro-biomass/train/ID1211362607.jpg
/kaggle/input/csiro-biomass/train/ID1853508321.jpg
/kaggle/input/csiro-biomass/train/ID193102215.jpg
/kaggle/input/csiro-biomass/train/ID698608346.jpg
/kaggle/input/csiro-biomass/train/ID1859251563.jpg
/kaggle/input/csiro-biomass/train/ID1880764911.jpg
/kaggle/input/csiro-biomass/train/ID853954911.jpg
/kaggle/input/csiro-biomass/train/ID1403107574.jpg
/kaggle/input/csiro-biomass/train/ID1781353117.jpg
/kaggle/input/csiro-biomass/train/ID384648061.jpg
/kaggle/input/csiro-biomass/train/ID1563418511.jpg
/kaggle/input/csiro-biomass/train/ID2125100696.jpg
/kaggle/input/csiro-biomass/train/ID482555369.jpg
/kaggle/input/csiro-biomass/train/ID638711343.jpg
/kaggle/input/c

In [2]:
# ============================================================
# CSIRO - Image2Biomass Prediction
# ============================================================

# -----------------------------
# 1. Imports & Setup
# -----------------------------
import os
import random
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

# Reproducibility
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -----------------------------
# 2. Paths
# -----------------------------
TRAIN_CSV = "/kaggle/input/csiro-biomass/train.csv"
TEST_CSV = "/kaggle/input/csiro-biomass/test.csv"
TRAIN_DIR = "/kaggle/input/csiro-biomass/train"
TEST_DIR = "/kaggle/input/csiro-biomass/test"

# -----------------------------
# 3. Load Data
# -----------------------------
train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

# Extract image_id from sample_id
train_df["image_id"] = train_df["sample_id"].str.split("__").str[0]
test_df["image_id"] = test_df["sample_id"].str.split("__").str[0]

# -----------------------------
# 4. Target Weights
# -----------------------------
TARGET_WEIGHTS = {
    "Dry_Green_g": 0.1,
    "Dry_Dead_g": 0.1,
    "Dry_Clover_g": 0.1,
    "GDM_g": 0.2,
    "Dry_Total_g": 0.5,
}

# -----------------------------
# 5. Image Dataset
# -----------------------------
IMG_SIZE = 224

image_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

class ImageDataset(Dataset):
    def __init__(self, image_paths):
        self.image_paths = image_paths

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert("RGB")
        return image_transforms(img)

# -----------------------------
# 6. CNN Feature Extractor
# -----------------------------
class CNNEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        base = models.resnet18(weights=None)  # offline-safe
        self.encoder = nn.Sequential(*list(base.children())[:-1])

    def forward(self, x):
        x = self.encoder(x)
        return x.view(x.size(0), -1)

encoder = CNNEncoder().to(DEVICE)
encoder.eval()

# -----------------------------
# 7. Feature Extraction
# -----------------------------
@torch.no_grad()
def extract_features(image_paths, batch_size=32):
    dataset = ImageDataset(image_paths)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    feats = []

    for batch in tqdm(loader, desc="Extracting CNN features"):
        batch = batch.to(DEVICE)
        feats.append(encoder(batch).cpu().numpy())

    return np.vstack(feats)

# -----------------------------
# 8. Train Image Features
# -----------------------------
train_images = train_df[["image_id", "image_path"]].drop_duplicates()
train_images["full_path"] = train_images["image_path"].apply(
    lambda x: os.path.join(TRAIN_DIR, os.path.basename(x))
)

train_img_features = extract_features(train_images["full_path"].values)

train_img_feat_df = pd.DataFrame(
    train_img_features,
    index=train_images["image_id"].values,
)

# -----------------------------
# 9. Tabular Features (TRAIN ONLY)
# -----------------------------
TAB_COLS = ["Pre_GSHH_NDVI", "Height_Ave_cm"]

train_tab = train_df.groupby("image_id")[TAB_COLS].mean()

scaler = StandardScaler()
train_tab_scaled = scaler.fit_transform(train_tab)

train_tab_df = pd.DataFrame(
    train_tab_scaled,
    index=train_tab.index,
    columns=TAB_COLS,
)

# -----------------------------
# 10. Merge Train Features
# -----------------------------
X_train_full = train_img_feat_df.join(train_tab_df)

# -----------------------------
# 11. Train Target-Specific Models
# -----------------------------
models_by_target = {}

for target in TARGET_WEIGHTS:
    print(f"\nTraining model for {target}")

    df_t = train_df[train_df["target_name"] == target]
    y = df_t.set_index("image_id").loc[X_train_full.index]["target"].values

    model = Ridge(alpha=10.0)
    model.fit(X_train_full.values, y)

    models_by_target[target] = model

# -----------------------------
# 12. Test Feature Extraction
# -----------------------------
test_images = test_df[["image_id", "image_path"]].drop_duplicates()
test_images["full_path"] = test_images["image_path"].apply(
    lambda x: os.path.join(TEST_DIR, os.path.basename(x))
)

test_img_features = extract_features(test_images["full_path"].values)

test_img_feat_df = pd.DataFrame(
    test_img_features,
    index=test_images["image_id"].values,
)

test_tab_df = pd.DataFrame(
    np.zeros((len(test_img_feat_df), len(TAB_COLS))),
    index=test_img_feat_df.index,
    columns=TAB_COLS,
)

X_test_full = test_img_feat_df.join(test_tab_df)

# -----------------------------
# 13. Generate Predictions
# -----------------------------
test_preds = []

for _, row in test_df.iterrows():
    img_id = row["image_id"]
    tname = row["target_name"]

    model = models_by_target[tname]
    pred = model.predict(X_test_full.loc[[img_id]].values)[0]

    test_preds.append(pred)

# -----------------------------
# 14. Submission
# -----------------------------
submission = pd.DataFrame({
    "sample_id": test_df["sample_id"],
    "target": test_preds
})

submission.to_csv("submission.csv", index=False)
print("\nSubmission saved as submission.csv")
submission.head()

Train shape: (1785, 9)
Test shape: (5, 3)


Extracting CNN features: 100%|██████████| 12/12 [00:21<00:00,  1.80s/it]



Training model for Dry_Green_g

Training model for Dry_Dead_g

Training model for Dry_Clover_g

Training model for GDM_g

Training model for Dry_Total_g


Extracting CNN features: 100%|██████████| 1/1 [00:00<00:00,  8.82it/s]


Submission saved as submission.csv


,sample_id,target
0,ID1001187975__Dry_Clover_g,7.497295
1,ID1001187975__Dry_Dead_g,16.819926
2,ID1001187975__Dry_Green_g,15.840951
3,ID1001187975__Dry_Total_g,40.154966
4,ID1001187975__GDM_g,23.338258
